In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col,trim
from pyspark.sql.types import StringType

RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "customer_bday",
    "GEN": "customer_gender",
}

In [0]:
df = spark.table("data_lakehouse_project.bronze.bronze_cust_az12")

#Data transform

Trim string value

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

Gender value handling

In [0]:
df = df.withColumn(
    "GEN",
    F.when(F.upper(F.col("GEN")) == "M", "Male")
     .when(F.upper(F.col("GEN")) == "F", "Female")
     .when((F.upper(F.col("GEN")) == "") | F.col("GEN").isNull(), "N/a")
     .otherwise(F.col("GEN"))
)


Remove unwanted string

In [0]:
df = df.withColumn(
    "CID", 
    F.when(
        F.col("CID").startswith("NAS"),        
        F.expr("substring(CID, 4, length(CID))")
    ).otherwise(F.col("CID"))
)

#Write to silver table cust_az12

In [0]:
# Renaming columns header
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("data_lakehouse_project.silver.silver_cust_az12")
)

In [0]:
%sql
select * from data_lakehouse_project.silver.silver_cust_az12